# Training Data Setup

This notebook prepares training, validation, and test datasets for ML models:
1. Load input (temperature) and target (CRE) data
2. Split by time periods (train/val/test)
3. Split by ensemble members
4. Create hemisphere-based splits for regional analysis
5. Save processed splits for model training

In [1]:
import sys
sys.path.append('..')

from utils import (
    load_dataset,
    select_variable
)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr
import rioxarray as rxr

## 1. Load Data

In [3]:
# Load input (surface temperature) and target (CRE) datasets
x_data = load_dataset("../../data/CanESM5_1850-2100_tas.nc")
y_data_rsut = load_dataset("../../data/CanESM5_1850-2100_rsutcre.nc")
y_data_rlut = load_dataset("../../data/CanESM5_1850-2100_rlutcre.nc")


In [5]:
# Select variables
x_var = select_variable(x_data, ["tas"])
y_var_rsut = select_variable(y_data_rsut, ["cre", "rsutcre"])
y_var_rlut = select_variable(y_data_rlut, ["cre", "rlutcre"])

x_da = x_data[x_var]
y_da_rsut = y_data_rsut[y_var_rsut]
y_da_rlut = y_data_rlut[y_var_rlut]

print(f"\nSelected variables:")
print(f"  Input: {x_var}")
print(f"  Label (RSUT):: {y_var_rsut}")
print(f"  Label (RLUT): {y_var_rlut}")


Selected variables:
  Input: tas
  Label (RSUT):: cre
  Label (RLUT): cre


In [6]:
# Align datasets to common coordinates
x_da, y_da_rsut, y_da_rlut = xr.align(x_da, y_da_rsut, y_da_rlut, join="inner")

print("Aligned datasets:")
print(f"  x: {x_da.dims}")
print(f"  y_rsut: {y_da_rsut.dims}")
print(f"  y_rlut: {y_da_rlut.dims}")

Aligned datasets:
  x: ('member', 'time', 'lat', 'lon')
  y_rsut: ('member', 'time', 'lat', 'lon')
  y_rlut: ('member', 'time', 'lat', 'lon')


## Time-Based Splitting

Split data into:
- **Train**: 1850-2014 (historical period)
- **Validation**: 1850-2014 (same period, different members)
- **Test**: 2015-2100 (future projections)

In [8]:
# Verify member dimension exists
if "member" not in x_da.dims or "member" not in y_da_rsut.dims or "member" not in y_da_rlut.dims:
    raise ValueError("Expected a 'member' dimension in input and both label datasets.")

n_members = x_da.sizes["member"]
if n_members != 25:
    raise ValueError(f"Expected 25 members, found {n_members}.")
print(f"Total ensemble members: {n_members}")

Total ensemble members: 25


In [9]:
# Create time masks
years = x_da["time"].dt.year
train_mask = (years >= 1850) & (years <= 2014)
val_mask = (years >= 1850) & (years <= 2014)  # Same period, different members
test_mask = years >= 2015

print("Time periods:")
print(f"  Train/Val: 1850-2014 ({train_mask.sum().values} timesteps)")
print(f"  Test: 2015+ ({test_mask.sum().values} timesteps)")

Time periods:
  Train/Val: 1850-2014 (1980 timesteps)
  Test: 2015+ (1032 timesteps)


In [10]:
# Random member split (reproducible)
rng = np.random.default_rng(42)
member_idx = rng.permutation(n_members)

train_idx = member_idx[:17]  # 68% for training
val_idx = member_idx[17:21]  # 16% for validation
test_idx = member_idx[21:]   # 16% for testing

print("\nMember splits:")
print(f"  Train: {len(train_idx)} members - {train_idx}")
print(f"  Val: {len(val_idx)} members - {val_idx}")
print(f"  Test: {len(test_idx)} members - {test_idx}")


Member splits:
  Train: 17 members - [15 16 19 20  9  7 17 24  6 10  3  0 21 18 12  5 11]
  Val: 4 members - [14 23  2  4]
  Test: 4 members - [22  1 13  8]


In [11]:
def subset_time(da, dim, mask):
    idx = np.where(mask.values)[0]
    return da.isel({dim: idx})

# Create train/val/test splits for input (temperature)
x_train_period = subset_time(x_da, "time", train_mask)
x_val_period = subset_time(x_da, "time", val_mask)
x_test_period = subset_time(x_da, "time", test_mask)

X_train = x_train_period.isel(member=train_idx)
X_val = x_val_period.isel(member=val_idx)
X_test = x_test_period.isel(member=test_idx)

print("\nInput (X) splits:")
print(f"  X_train: {X_train.shape}")
print(f"  X_val: {X_val.shape}")
print(f"  X_test: {X_test.shape}")


Input (X) splits:
  X_train: (17, 1980, 64, 128)
  X_val: (4, 1980, 64, 128)
  X_test: (4, 1032, 64, 128)


In [12]:
# Create train/val/test splits for targets (CRE)
# Shortwave CRE
y_rsut_train_period = subset_time(y_da_rsut, "time", train_mask)
y_rsut_val_period = subset_time(y_da_rsut, "time", val_mask)
y_rsut_test_period = subset_time(y_da_rsut, "time", test_mask)

y_train_rsut = y_rsut_train_period.isel(member=train_idx)
y_val_rsut = y_rsut_val_period.isel(member=val_idx)
y_test_rsut = y_rsut_test_period.isel(member=test_idx)

# Longwave CRE
y_rlut_train_period = subset_time(y_da_rlut, "time", train_mask)
y_rlut_val_period = subset_time(y_da_rlut, "time", val_mask)
y_rlut_test_period = subset_time(y_da_rlut, "time", test_mask)

y_train_rlut = y_rlut_train_period.isel(member=train_idx)
y_val_rlut = y_rlut_val_period.isel(member=val_idx)
y_test_rlut = y_rlut_test_period.isel(member=test_idx)

print("\nTarget (y) splits - Shortwave CRE:")
print(f"  y_train_rsut: {y_train_rsut.shape}")
print(f"  y_val_rsut: {y_val_rsut.shape}")
print(f"  y_test_rsut: {y_test_rsut.shape}")

print("\nTarget (y) splits - Longwave CRE:")
print(f"  y_train_rlut: {y_train_rlut.shape}")
print(f"  y_val_rlut: {y_val_rlut.shape}")
print(f"  y_test_rlut: {y_test_rlut.shape}")


Target (y) splits - Shortwave CRE:
  y_train_rsut: (17, 1980, 64, 128)
  y_val_rsut: (4, 1980, 64, 128)
  y_test_rsut: (4, 1032, 64, 128)

Target (y) splits - Longwave CRE:
  y_train_rlut: (17, 1980, 64, 128)
  y_val_rlut: (4, 1980, 64, 128)
  y_test_rlut: (4, 1032, 64, 128)


## Period-Based Splits

Create additional splits by time period for temporal generalization analysis:
- **Past**: 1850-1950
- **Present**: 1950-2015
- **Future**: 2015-2100

In [17]:
# ============================================================================
# Period-Based Data Splits (Past, Present, Future)
# ============================================================================

# Define time dimension (check for common names across datasets)
time_dim_candidates = ["time", "year", "date"]
time_dim = next(
    (d for d in time_dim_candidates 
     if d in x_da.dims and d in y_da_rsut.dims and d in y_da_rlut.dims), 
    None
)

if time_dim is None:
    raise ValueError(
        "Could not find a shared time dimension (expected one of: time/year/date)."
    )

# Create time masks for three climate periods
mask_past = (years >= 1850) & (years < 1950)      # Historical period
mask_present = (years >= 1950) & (years < 2015)   # Observational data
mask_future = (years >= 2015) & (years <= 2100)   # Future projections

# Split input data (temperature) by time period
X_past = subset_time(x_da, time_dim, mask_past)
X_present = subset_time(x_da, time_dim, mask_present)
X_future = subset_time(x_da, time_dim, mask_future)

# Split shortwave CRE targets by time period
y_rsut_past = subset_time(y_da_rsut, time_dim, mask_past)
y_rsut_present = subset_time(y_da_rsut, time_dim, mask_present)
y_rsut_future = subset_time(y_da_rsut, time_dim, mask_future)

# Split longwave CRE targets by time period
y_rlut_past = subset_time(y_da_rlut, time_dim, mask_past)
y_rlut_present = subset_time(y_da_rlut, time_dim, mask_present)
y_rlut_future = subset_time(y_da_rlut, time_dim, mask_future)

# Organize period splits into dictionary for easy access
period_splits = {
    "past_1850_1950": {
        "X": X_past, 
        "y_rsut": y_rsut_past, 
        "y_rlut": y_rlut_past
    },
    "present_1950_2015": {
        "X": X_present, 
        "y_rsut": y_rsut_present, 
        "y_rlut": y_rlut_present
    },
    "future_2015_2100": {
        "X": X_future, 
        "y_rsut": y_rsut_future, 
        "y_rlut": y_rlut_future
    },
}

# Print summary of period splits
print(f"Time dimension used: {time_dim}")
print("\nPeriod splits created:")
for split_name, split_data in period_splits.items():
    n_samples = split_data["X"].sizes[time_dim]
    print(f"  {split_name}: {n_samples} samples")

Time dimension used: time

Period splits created:
  past_1850_1950: 1200 samples
  present_1950_2015: 780 samples
  future_2015_2100: 1032 samples


## Hemisphere-Based Splits

Create regional splits for analyzing model performance by latitude:
- **Northern**: lat > 23.5°N
- **Equatorial**: -23.5° ≤ lat ≤ 23.5°
- **Southern**: lat < -23.5°S

I Would suggest to do it at 30

In [19]:
# ============================================================================
# Hemisphere-Based Data Splits (Northern, Equatorial, Southern)
# ============================================================================

# Find the latitude dimension (check for common names across datasets)
lat_dim_candidates = ["lat", "latitude", "y"]
lat_dim = next(
    (d for d in lat_dim_candidates 
     if d in x_da.dims and d in y_da_rsut.dims and d in y_da_rlut.dims), 
    None
)

if lat_dim is None:
    raise ValueError(
        "Could not find a shared latitude dimension (expected one of: lat/latitude/y)."
    )

# Extract latitude values
lat_vals = x_da[lat_dim]

# Define latitude thresholds for regional splits (using 30° for cleaner boundaries)
lat_threshold = 30.0

mask_north = lat_vals > lat_threshold           # Northern: lat > 30°N
mask_equator = (lat_vals >= -lat_threshold) & (lat_vals <= lat_threshold)  # Equatorial: -30° to 30°
mask_south = lat_vals < -lat_threshold           # Southern: lat < 30°S

def subset_lat(da, mask):
    """Select data along latitude dimension using a boolean mask."""
    return da.where(mask, drop=True)

# Split input data (temperature) by latitude region
X_north = subset_lat(x_da, mask_north)
X_equator = subset_lat(x_da, mask_equator)
X_south = subset_lat(x_da, mask_south)

# Split shortwave CRE targets by latitude region
y_rsut_north = subset_lat(y_da_rsut, mask_north)
y_rsut_equator = subset_lat(y_da_rsut, mask_equator)
y_rsut_south = subset_lat(y_da_rsut, mask_south)

# Split longwave CRE targets by latitude region
y_rlut_north = subset_lat(y_da_rlut, mask_north)
y_rlut_equator = subset_lat(y_da_rlut, mask_equator)
y_rlut_south = subset_lat(y_da_rlut, mask_south)

# Organize hemisphere splits into dictionary for easy access
hemisphere_splits = {
    "northern": {"X": X_north, "y_rsut": y_rsut_north, "y_rlut": y_rlut_north},
    "equator_band": {"X": X_equator, "y_rsut": y_rsut_equator, "y_rlut": y_rlut_equator},
    "southern": {"X": X_south, "y_rsut": y_rsut_south, "y_rlut": y_rlut_south},
}

# Print summary of hemisphere splits
print(f"Latitude dimension used: {lat_dim}")
print(f"Latitude threshold: ±{lat_threshold}°")
print("\nHemisphere splits created:")
for split_name, split_data in hemisphere_splits.items():
    n_lats = split_data["X"].sizes[lat_dim]
    print(f"  {split_name}: {n_lats} latitude points")

Latitude dimension used: lat
Latitude threshold: ±30.0°

Hemisphere splits created:
  northern: 21 latitude points
  equator_band: 22 latitude points
  southern: 21 latitude points


## Save Processed Splits

Save all splits for use in model training.

We need to create the function to save the datasets
`save_dataset()`

In [ ]:
# Save main train/val/test splits
output_dir = "../../data/splits/"

# Input splits
save_dataset(X_train, f"{output_dir}X_train.nc")
save_dataset(X_val, f"{output_dir}X_val.nc")
save_dataset(X_test, f"{output_dir}X_test.nc")

# Shortwave CRE targets
save_dataset(y_train_rsut, f"{output_dir}y_train_rsut.nc")
save_dataset(y_val_rsut, f"{output_dir}y_val_rsut.nc")
save_dataset(y_test_rsut, f"{output_dir}y_test_rsut.nc")

# Longwave CRE targets
save_dataset(y_train_rlut, f"{output_dir}y_train_rlut.nc")
save_dataset(y_val_rlut, f"{output_dir}y_val_rlut.nc")
save_dataset(y_test_rlut, f"{output_dir}y_test_rlut.nc")

print("Saved main train/val/test splits")

In [ ]:
# Save period splits
for period_name, data in period_splits.items():
    save_dataset(data['X'], f"{output_dir}period_{period_name}_X.nc")
    save_dataset(data['y_rsut'], f"{output_dir}period_{period_name}_y_rsut.nc")
    save_dataset(data['y_rlut'], f"{output_dir}period_{period_name}_y_rlut.nc")

print("Saved period splits")

In [ ]:
# Save hemisphere splits
for region_name, data in hemisphere_splits.items():
    save_dataset(data['X'], f"{output_dir}hemisphere_{region_name}_X.nc")
    save_dataset(data['y_rsut'], f"{output_dir}hemisphere_{region_name}_y_rsut.nc")
    save_dataset(data['y_rlut'], f"{output_dir}hemisphere_{region_name}_y_rlut.nc")

print("Saved hemisphere splits")